In [ ]:
def validate_heatmap_inputs(results, metrics, model_order):
    required_columns = {"threshold", "model", *metrics}
    missing_columns = sorted(required_columns - set(results.columns))
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    if results["model"].nunique() != 2:
        raise ValueError("This heatmap requires exactly two models.")

    if model_order is not None and set(model_order) != set(results["model"].unique()):
        raise ValueError("model_order must match the two model names in results.")

    duplicate_rows = results.duplicated(subset=["threshold", "model"]).any()
    if duplicate_rows:
        raise ValueError("Each threshold-model combination must appear only once.")


def resolve_model_order(results, model_order=None):
    if model_order is not None:
        return model_order
    return sorted(results["model"].unique().tolist())


def build_metric_table(results, metric, model_order):
    metric_table = results.pivot(
        index="threshold",
        columns="model",
        values=metric,
    )
    return metric_table[model_order].sort_index()


def choose_winner_from_values(value_a, value_b, model_a, model_b, higher_is_better):
    if pd.isna(value_a) and pd.isna(value_b):
        return "Tie"
    if pd.isna(value_a):
        return model_b
    if pd.isna(value_b):
        return model_a
    if np.isclose(value_a, value_b):
        return "Tie"

    if higher_is_better:
        return model_a if value_a > value_b else model_b
    return model_a if value_a < value_b else model_b


def build_winner_table(results, metrics, metric_directions, model_order):
    winner_columns = {}

    for metric in metrics:
        metric_table = build_metric_table(results, metric, model_order)
        model_a, model_b = model_order

        winners = metric_table.apply(
            lambda row: choose_winner_from_values(
                value_a=row[model_a],
                value_b=row[model_b],
                model_a=model_a,
                model_b=model_b,
                higher_is_better=metric_directions[metric],
            ),
            axis=1,
        )
        winner_columns[metric] = winners

    winner_table = pd.DataFrame(winner_columns)
    winner_table.index.name = "threshold"
    return winner_table


def build_annotation_table(results, metrics, model_order, model_labels=None):
    model_labels = model_labels or {}
    annotations = {}

    for metric in metrics:
        metric_table = build_metric_table(results, metric, model_order)
        model_a, model_b = model_order
        label_a = model_labels.get(model_a, model_a)
        label_b = model_labels.get(model_b, model_b)

        annotations[metric] = metric_table.apply(
            lambda row: f"{label_a}: {row[model_a]:.3f}\n{label_b}: {row[model_b]:.3f}",
            axis=1,
        )

    annotation_table = pd.DataFrame(annotations)
    annotation_table.index.name = "threshold"
    return annotation_table


def prettify_metric_labels(metrics):
    return [metric.replace("_", " ").title() for metric in metrics]


def plot_threshold_winner_heatmap(
    results,
    metrics,
    metric_directions,
    title="Which Model Performs Better Across Decision Thresholds?",
    model_order=None,
    model_labels=None,
    metric_labels=None,
    output_path=None,
    figsize=(12, 7),
):
    validate_heatmap_inputs(results, metrics, model_order)
    model_order = resolve_model_order(results, model_order)
    model_a, model_b = model_order

    winner_table = build_winner_table(
        results=results,
        metrics=metrics,
        metric_directions=metric_directions,
        model_order=model_order,
    )

    annotation_table = build_annotation_table(
        results=results,
        metrics=metrics,
        model_order=model_order,
        model_labels=model_labels,
    )

    code_map = {model_a: 0, model_b: 1, "Tie": 2}
    encoded = winner_table.replace(code_map).astype(int)

    cmap = ListedColormap(["#4C78A8", "#F58518", "#BDBDBD"])
    norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], cmap.N)

    sns.set_theme(style="white", font_scale=0.95)
    fig, ax = plt.subplots(figsize=figsize)

    sns.heatmap(
        encoded,
        annot=annotation_table,
        fmt="",
        cmap=cmap,
        norm=norm,
        cbar=False,
        linewidths=1,
        linecolor="white",
        annot_kws={"fontsize": 9, "fontweight": "semibold"},
        ax=ax,
    )

    display_labels = model_labels or {}
    x_labels = metric_labels if metric_labels is not None else prettify_metric_labels(metrics)
    y_labels = [f"{float(threshold):.2%}" for threshold in encoded.index]

    ax.set_title(title, fontsize=14, fontweight="bold", pad=14)
    ax.set_xlabel("Metric", fontsize=11)
    ax.set_ylabel("Decision Threshold", fontsize=11)
    ax.set_xticklabels(x_labels, rotation=0, ha="center")
    ax.set_yticklabels(y_labels, rotation=0)

    ax.legend(
        handles=[
            mpatches.Patch(color="#4C78A8", label=f"{display_labels.get(model_a, model_a)} wins"),
            mpatches.Patch(color="#F58518", label=f"{display_labels.get(model_b, model_b)} wins"),
            mpatches.Patch(color="#BDBDBD", label="Tie"),
        ],
        title="Better Model",
        loc="upper center",
        bbox_to_anchor=(0.5, -0.12),
        ncol=3,
        frameon=False,
    )

    fig.tight_layout()

    if output_path is not None:
        fig.savefig(output_path, dpi=300, bbox_inches="tight")

    return winner_table, annotation_table, fig
